In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

print("1. Загрузка тестовых данных и моделей...")
df_test_raw = pd.read_csv('test_real_estate_samples.csv', encoding='utf-8')
df_original = df_test_raw.copy()

# Загружаем обе модели
model_sale = joblib.load('best_model_sale.pkl')
model_rent = joblib.load('best_model_rent.pkl')
scaler_sale = joblib.load('scaler_sale.pkl')
scaler_rent = joblib.load('scaler_rent.pkl')
feature_cols = joblib.load('feature_cols.pkl')

print(f"Загружено {len(df_test_raw)} объектов для тестирования.")

# Функция для предобработки одного куска данных
def preprocess_data(df_chunk):
    df = df_chunk.copy()
    df = df.drop(columns=['furnished', 'end_date', 'id', 'price'], errors='ignore')

    room_mapping = {'1+0': 0, '1+1': 1, '2+1': 2, '2+2': 3, '3+1': 4, '3+2': 5,
                    '4+1': 6, '4+2': 7, '5+1': 8, '5+2': 9, '+': 10}
    df['room_count_enc'] = df['room_count'].map(room_mapping).fillna(10)

    def parse_age(x):
        if pd.isna(x): return np.nan
        x = str(x)
        if x == '0': return 0
        if 've üzeri' in x: return int(x.split()[0]) + 5
        if '-' in x:
            parts = x.split('-')
            return (int(parts[0]) + int(parts[1].split()[0])) / 2
        try: return float(x)
        except: return np.nan
    df['building_age_num'] = df['building_age'].apply(parse_age)

    def parse_floor(x):
        if pd.isna(x): return np.nan
        x = str(x)
        if 've üzeri' in x: return int(x.split()[0]) + 5
        if 'Yüksek Giriş' in x or 'Zemin' in x: return 0
        try: return float(x)
        except: return np.nan
    df['floor_no_num'] = df['floor_no'].apply(parse_floor)
    df['total_floor_num'] = df['total_floor_count'].apply(parse_floor)

    df['district'] = df['address'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else 'Unknown')

    nominal_cols = ['type', 'sub_type', 'heating_type', 'price_currency', 'district']
    df = pd.get_dummies(df, columns=nominal_cols, prefix=nominal_cols, drop_first=False)

    central_districts = ['Beşiktaş', 'Şişli', 'Kadıköy', 'Bakırköy', 'Ataşehir', 'Levent', 'Konak']
    central_cols = [c for c in df.columns if any(d in c for d in central_districts)]
    df['is_central'] = df[central_cols].sum(axis=1).clip(upper=1) if len(central_cols) > 0 else 0

    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0

    return df[feature_cols].copy()

print("\n2. Разделение данных и предобработка...")
# Разделяем на продажу и аренду
df_sale = df_test_raw[df_test_raw['listing_type'] == 1].copy()
df_rent = df_test_raw[df_test_raw['listing_type'] == 2].copy()

print(f"Объектов продажи: {len(df_sale)}, Объектов аренды: {len(df_rent)}")

X_sale = preprocess_data(df_sale)
X_rent = preprocess_data(df_rent)

print("\n3. Прогнозирование...")
# Предсказываем для продажи
X_sale_scaled = scaler_sale.transform(X_sale)
pred_log_sale = model_sale.predict(X_sale_scaled)
pred_real_sale = np.expm1(pred_log_sale)

# Предсказываем для аренды
X_rent_scaled = scaler_rent.transform(X_rent)
pred_log_rent = model_rent.predict(X_rent_scaled)
pred_real_rent = np.expm1(pred_log_rent)

# Собираем предсказания обратно в исходный порядок
predictions = np.zeros(len(df_original))
predictions[df_original['listing_type'] == 1] = pred_real_sale
predictions[df_original['listing_type'] == 2] = pred_real_rent

df_original['Predicted_Price'] = predictions

print("\n4. Расчет метрик...")
y_true = df_original['price'].values
y_pred = df_original['Predicted_Price'].values

mae_lib = mean_absolute_error(y_true, y_pred)
rmse_lib = np.sqrt(mean_squared_error(y_true, y_pred))
r2_lib = r2_score(y_true, y_pred)

print(f"MAE (библиотека): {mae_lib:,.2f} TRY")
print(f"RMSE (библиотека): {rmse_lib:,.2f} TRY")
print(f"R2 (библиотека): {r2_lib:.4f}")

mae_manual = np.mean(np.abs(y_true - y_pred))
rmse_manual = np.sqrt(np.mean((y_true - y_pred)**2))
y_mean = np.mean(y_true)
r2_manual = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - y_mean)**2))

print(f"\nMAE (вручную): {mae_manual:,.2f} TRY")
print(f"RMSE (вручную): {rmse_manual:,.2f} TRY")
print(f"R2 (вручную): {r2_manual:.4f}")

print("\n5. Сохранение результата...")
df_original['ФИО'] = 'Жеребов Даниил Артёмович'
df_original['Группа'] = '23П-2'

output_filename = 'result_real_estate_samples.csv'
df_original.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"Файл успешно сохранен: {output_filename}")

print("\nИтоговая таблица:")
display(df_original[['id', 'listing_type', 'address', 'room_count', 'size', 'price', 'Predicted_Price']].head(10))

1. Загрузка тестовых данных и моделей...
Загружено 20 объектов для тестирования.

2. Разделение данных и предобработка...
Объектов продажи: 10, Объектов аренды: 10

3. Прогнозирование...

4. Расчет метрик...
MAE (библиотека): 206,380.74 TRY
RMSE (библиотека): 507,051.97 TRY
R2 (библиотека): 0.3399

MAE (вручную): 206,380.74 TRY
RMSE (вручную): 507,051.97 TRY
R2 (вручную): 0.3399

5. Сохранение результата...
Файл успешно сохранен: result_real_estate_samples.csv

Итоговая таблица:


,id,listing_type,address,room_count,size,price,Predicted_Price
0,49,1,İzmir/Aliağa/Yenişakran,1+0,1.0,300000.0,246062.799881
1,506,1,İzmir/Foça/Yeniköy,2+2,250.0,1600000.0,728222.422848
2,2,1,İstanbul/Kartal/Kordonboyu,1+0,43.0,490000.0,215249.700248
3,528,1,Tekirdağ/Çerkezköy/Kızılpınar,3+1,122.0,103600.0,110645.525759
4,109,1,İzmir/Aliağa/Yenişakran,1+0,1.0,300000.0,250663.766914
5,100,1,Mersin/Erdemli/Kargıpınarı,3+1,150.0,500000.0,539402.983488
6,320,1,Balıkesir/Burhaniye/Pelitköy,4+1,134.0,1000000.0,461399.547300
7,3,1,Tekirdağ/Çorlu/Reşadiye,2+1,NaN,155000.0,216514.074524
8,7810,1,İzmir/Konak/Akın Simav,4+1,145.0,650000.0,465474.633721
9,11,1,Muğla/Bodrum/Ortakentyahşi,3+1,125.0,2450000.0,457265.173911
